In [ ]:

import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("../dataset/womens_clothing_analysis.csv")
df = df.dropna(subset=["Review", "Rating"])

df["label"] = (df["Rating"] >= 4).astype(int)


train_df, test_df = train_test_split(
    df[["Review", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_data = Dataset.from_pandas(train_df.reset_index(drop=True))
test_data = Dataset.from_pandas(test_df.reset_index(drop=True))

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


def tokenize_reviews(batch):
    return tokenizer(
        batch["Review"],
        truncation=True,
        max_length=128
    )

train_data = train_data.map(
    tokenize_reviews,
    batched=True,
    remove_columns=["Review"]
)

test_data = test_data.map(
    tokenize_reviews,
    batched=True,
    remove_columns=["Review"]
)


training_args = TrainingArguments(
    output_dir="./sentiment_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

predictions = trainer.predict(test_data)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Positive"],
    zero_division=0
))


model.save_pretrained("./clothing_sentiment_model")
tokenizer.save_pretrained("./clothing_sentiment_model")

print("\nModel saved successfully!")

def predict_sentiment(review):
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

    label = probabilities.argmax(dim=1).item()
    confidence = probabilities[0, label].item() * 100

    sentiment = "Positive" if label == 1 else "Negative"

    return sentiment, round(confidence, 2)


review = "The dress is beautiful and very comfortable."
sentiment, confidence = predict_sentiment(review)

print("\nReview:", review)
print("Sentiment:", sentiment)
print("Confidence:", confidence, "%")



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/536 [00:00<?, ? examples/s]

Map:   0%|          | 0/134 [00:00<?, ? examples/s]

C:\Users\MAYUR\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.472755
2,No log,0.452202
3,No log,0.406981


Accuracy: 0.746268656716418

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.61      0.69        61
    Positive       0.72      0.86      0.79        73

    accuracy                           0.75       134
   macro avg       0.76      0.73      0.74       134
weighted avg       0.75      0.75      0.74       134



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!

Review: The dress is beautiful and very comfortable.
Sentiment: Positive
Confidence: 70.2 %
